# DocuVision AI — Training Notebook (run on Colab, free T4 GPU)

Runtime > Change runtime type > **T4 GPU** before running this.

In [2]:
!nvidia-smi

Tue Jul 21 18:58:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/DocuVisionAI"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_PROJECT_DIR}")

Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/DocuVisionAI


In [4]:
from typing_extensions import Doc
%cd /content
!rm -rf DocuVisionAI
!git clone https://github.com/fathimarfa/DocuVisionAI.git
%cd DocuVisionAI

/content
Cloning into 'DocuVisionAI'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 97 (delta 42), reused 71 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 29.53 KiB | 4.92 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/DocuVisionAI


In [5]:
!cat requirements.txt

transformers==4.46.3
sentencepiece>=0.1.99
datasets>=2.19.0
Pillow>=10.1.0
jiwer>=3.0.0
accelerate>=0.24.0
tensorboard>=2.15.0
PyYAML>=6.0.1
tqdm>=4.66.1
pandas>=2.1.3
numpy>=1.26.2
scikit-learn>=1.3.2
pytest>=7.4.3

In [6]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 92.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [7]:
# 2. Download + split the dataset
!python data/download_dataset.py --config configs/trocr_base.yaml

2026-07-21 19:03:23,060 [INFO] Downloading dataset: Teklia/IAM-line
README.md: 2.14kB [00:00, 4.08MB/s]
2026-07-21 19:03:24,672 [INFO] Available splits: ['train', 'validation', 'test']
data/train.parquet: 100% 167M/167M [00:03<00:00, 45.5MB/s]
data/validation.parquet: 100% 24.7M/24.7M [00:00<00:00, 31.9MB/s]
data/test.parquet: 100% 73.6M/73.6M [00:01<00:00, 57.9MB/s]
Generating train split: 100% 6482/6482 [00:00<00:00, 13780.23 examples/s]
Generating validation split: 100% 976/976 [00:00<00:00, 15805.44 examples/s]
Generating test split: 100% 2915/2915 [00:00<00:00, 16125.45 examples/s]
2026-07-21 19:04:00,702 [INFO] Wrote train (500 rows) -> data/processed/train.csv
2026-07-21 19:04:06,146 [INFO] Wrote val (97 rows) -> data/processed/val.csv
2026-07-21 19:04:22,073 [INFO] Wrote test (291 rows) -> data/processed/test.csv


In [8]:
# 3. Fine-tune TrOCR (mixed precision + gradient accumulation configured in YAML)
!python -m src.train --config configs/trocr_base.yaml

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
2026-07-21 19:04:39.184916: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
preprocessor_config.json: 100% 224/224 [00:00<00:00, 1.90MB/s]
tokenizer_config.json: 1.12kB [00:00, 5.89MB/s]
vocab.json: 899kB [00:00, 49.5MB/s]
merges.txt: 456kB [00:00, 108MB/s]
special_tokens_map.json: 100% 772/772 [00:00<00:00, 7.02MB/s]
config.json: 4.17kB [00:00, 19.5MB/s]
model.safetensors: 100% 1.33G/1.33G [00:12<00:00, 106MB/s]
Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwr

In [9]:
!ls weights/checkpoint-final

config.json		preprocessor_config.json  training_args.bin
generation_config.json	special_tokens_map.json   vocab.json
merges.txt		tokenizer_config.json
model.safetensors	tokenizer.json


In [10]:
# 4. Evaluate: fine-tuned vs. base model zero-shot (CER / WER report)
!python -m src.evaluate --checkpoint weights/checkpoint-final --base-model microsoft/trocr-base-handwritten

2026-07-21 19:13:39.864391: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder 

In [ ]:
!python -m src.inference --checkpoint weights/checkpoint-best --image path/to/image.png

In [13]:
# 5. Zip the best checkpoint and download it locally (for use in VS Code / app.py)
!zip -r checkpoint-best.zip weights/checkpoint-best
from google.colab import files
files.download('checkpoint-best.zip')

updating: weights/checkpoint-best/ (stored 0%)
updating: weights/checkpoint-best/.gitkeep (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 6. (Optional) push the trained checkpoint + results back to your repo
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"
# !git add weights/checkpoint-best
# !git commit -m "Add trained checkpoint"
# !git push